In [1]:
import sys
sys.path.append("../src")

import matplotlib.pyplot as plt
import seaborn as sns
from fonctions import *

btc = charger_donnees("BTC-USD", "2018-01-01", "2026-11-01")
eth = charger_donnees("ETH-USD", "2018-01-01", "2026-11-01")

btc.to_csv("../data/btc_brut.csv")
eth.to_csv("../data/eth_brut.csv")

print(btc.shape, eth.shape)
btc.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

(3226, 5) (3226, 5)


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-01,13657.200195,14112.200195,13154.700195,14112.200195,10291200000
2018-01-02,14982.099609,15444.599609,13163.599609,13625.000000,16846600192
2018-01-03,15201.000000,15572.799805,14844.500000,14978.200195,16871900160
2018-01-04,15599.200195,15739.700195,14522.200195,15270.700195,21783199744
2018-01-05,17429.500000,17705.199219,15202.799805,15477.200195,23840899072


In [5]:
with open("../src/fonctions.py", encoding="utf-8") as f:
    texte = f.read()

print("charger_donnees_8h présente ?", "def charger_donnees_8h" in texte)
print("Début du fichier :", texte[:30])

charger_donnees_8h présente ? True
Début du fichier : ﻿import numpy as np
import pan


In [3]:
code = '''import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler


def charger_donnees(symbole, debut, fin):
    """Télécharge les données OHLCV journalières depuis Yahoo Finance."""
    df = yf.download(symbole, start=debut, end=fin, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.dropna()
    return df


def charger_donnees_8h(symbole="BTCUSDT", debut="2018-01-01", fin="2026-11-01"):
    """Télécharge les bougies de 8h depuis Binance (3 lignes par jour)."""
    import requests, time
    url = "https://api.binance.com/api/v3/klines"
    t = int(pd.Timestamp(debut, tz="UTC").timestamp() * 1000)
    t_fin = int(pd.Timestamp(fin, tz="UTC").timestamp() * 1000)
    lignes = []
    while t < t_fin:
        r = requests.get(url, params={"symbol": symbole, "interval": "8h",
                                      "startTime": t, "endTime": t_fin,
                                      "limit": 1000}, timeout=30)
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        lignes += data
        t = data[-1][0] + 1
        time.sleep(0.2)
    df = pd.DataFrame(lignes, columns=[
        "Date", "Open", "High", "Low", "Close", "Volume", "close_time",
        "quote_volume", "trades", "taker_base", "taker_quote", "ignore"])
    df["Date"] = pd.to_datetime(df["Date"], unit="ms")
    df = df.set_index("Date")[["Close", "High", "Low", "Open", "Volume"]].astype(float)
    return df


def calculer_rendement_log(prix):
    """Rendement log : r_t = ln(P_t / P_{t-1})."""
    return np.log(prix / prix.shift(1))


def calculer_volatilite(rendements, fenetre=20):
    """Écart-type des rendements sur une fenêtre glissante."""
    return rendements.rolling(window=fenetre).std()


def calculer_RSI(prix, periode=14):
    """RSI : indicateur entre 0 et 100."""
    variation = prix.diff()
    gain = variation.clip(lower=0)
    perte = -variation.clip(upper=0)

    gain_moyen = gain.ewm(alpha=1/periode, min_periods=periode).mean()
    perte_moyenne = perte.ewm(alpha=1/periode, min_periods=periode).mean()

    rs = gain_moyen / perte_moyenne
    return 100 - (100 / (1 + rs))


def calculer_MACD(prix, court=12, long=26, signal=9):
    """MACD = EMA(12) - EMA(26). Signal = EMA(9) du MACD."""
    ema_court = prix.ewm(span=court, adjust=False).mean()
    ema_long = prix.ewm(span=long, adjust=False).mean()
    macd = ema_court - ema_long
    ligne_signal = macd.ewm(span=signal, adjust=False).mean()
    histogramme = macd - ligne_signal
    return macd, ligne_signal, histogramme


def standardiser(X):
    """Centre et réduit : moyenne 0, écart-type 1."""
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    return X_std, scaler
'''

with open("../src/fonctions.py", "w", encoding="utf-8") as f:
    f.write(code)

print("Fichier réécrit :", len(code), "caractères")

Fichier réécrit : 2817 caractères


In [5]:
with open("../src/fonctions.py", encoding="utf-8") as f:
    texte = f.read()

print("@' dans le fichier ?", "@'" in texte)
print("Nb de def charger_donnees_8h :", texte.count("def charger_donnees_8h"))
print("Début :", repr(texte[:20]))

@' dans le fichier ? False
Nb de def charger_donnees_8h : 1
Début : 'import numpy as np\ni'


In [7]:
import sys
sys.path.append("../src")
from fonctions import *

btc8 = charger_donnees_8h("BTCUSDT", "2018-01-01", "2026-11-01")
eth8 = charger_donnees_8h("ETHUSDT", "2018-01-01", "2026-11-01")

btc8.to_csv("../data/btc_8h.csv")
eth8.to_csv("../data/eth_8h.csv")

print(btc8.shape, eth8.shape)
btc8.head(6)

(9676, 5) (9676, 5)


,Close,High,Low,Open,Volume
Date,,,,,
2018-01-01 00:00:00,13570.35,13818.55,13155.38,13715.65,2978.419643
2018-01-01 08:00:00,13247.00,13735.24,12750.00,13569.98,3151.689084
2018-01-01 16:00:00,13380.00,13599.70,12940.00,13247.00,2479.807117
2018-01-02 00:00:00,13343.00,13850.00,12890.02,13382.16,5345.848979
2018-01-02 08:00:00,13690.03,13894.86,13302.59,13343.01,5962.062182
2018-01-02 16:00:00,14675.11,15473.49,13503.00,13690.02,8770.180950


In [9]:
code = '''

def charger_binance(symbole="BTCUSDT", intervalle="8h", debut="2017-08-17", fin="2026-11-01"):
    """Télécharge des bougies Binance (1h, 4h, 8h, 1d...) avec pagination."""
    import requests, time
    url = "https://api.binance.com/api/v3/klines"
    t = int(pd.Timestamp(debut, tz="UTC").timestamp() * 1000)
    t_fin = int(pd.Timestamp(fin, tz="UTC").timestamp() * 1000)
    lignes = []
    while t < t_fin:
        r = requests.get(url, params={"symbol": symbole, "interval": intervalle,
                                      "startTime": t, "endTime": t_fin,
                                      "limit": 1000}, timeout=30)
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        lignes += data
        t = data[-1][0] + 1
        time.sleep(0.2)
    df = pd.DataFrame(lignes, columns=[
        "Date", "Open", "High", "Low", "Close", "Volume", "close_time",
        "quote_volume", "trades", "taker_base", "taker_quote", "ignore"])
    df["Date"] = pd.to_datetime(df["Date"], unit="ms")
    df = df.set_index("Date")[["Close", "High", "Low", "Open", "Volume"]].astype(float)
    return df
'''

with open("../src/fonctions.py", "a", encoding="utf-8") as f:
    f.write(code)

print("Fonction ajoutée")

Fonction ajoutée


In [3]:
import sys
sys.path.append("../src")
from fonctions import *

cryptos = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]

donnees = {}
for c in cryptos:
    df = charger_binance(c, "8h", "2017-08-17", "2026-11-01")
    df.to_csv(f"../data/{c.lower()}_8h.csv")
    donnees[c] = df
    print(c, df.shape, "| début :", df.index[0])

BTCUSDT (10087, 5) | début : 2017-08-17 00:00:00
ETHUSDT (10087, 5) | début : 2017-08-17 00:00:00
BNBUSDT (9844, 5) | début : 2017-11-06 00:00:00
SOLUSDT (6820, 5) | début : 2020-08-11 00:00:00
XRPUSDT (9309, 5) | début : 2018-05-04 08:00:00
ADAUSDT (9361, 5) | début : 2018-04-17 00:00:00


In [5]:
btc1h = charger_binance("BTCUSDT", "1h", "2018-01-01", "2026-11-01")
btc1h.to_csv("../data/btcusdt_1h.csv")
print(btc1h.shape)     # environ 78 000 lignes

(77303, 5)
